# Pump It Up: Data Mining the Water Table
### Íñigo Ochoa — DrivenData Competition #7

**Resultado final:** Accuracy **0.8186** · Posición **2624 / 19 775** (top ~12%)

---

**Enfoque:**
1. EDA y limpieza de datos
2. Feature Engineering (con correcta separación train/validación)
3. Búsqueda de hiperparámetros (GridSearchCV / RandomizedSearchCV)
4. Ensemble de Soft Voting: Random Forest + LightGBM + XGBoost (pesos 3-2-2)
5. Validación cruzada estratificada (StratifiedKFold, 5 folds) y tracking con MLflow


## 1. Instalación de dependencias e importación de librerías

In [ ]:
# Colab: montar Google Drive
from google.colab import drive
drive.mount('/content/main')


In [ ]:
!pip install -q ydata_profiling lightgbm mlflow pyngrok


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from ydata_profiling import ProfileReport

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.model_selection import (
    StratifiedKFold, cross_val_score, cross_val_predict,
    GridSearchCV, RandomizedSearchCV
)
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

import mlflow
import mlflow.sklearn


## 2. Configuración

In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 2000)

# ---------------------------------------------------------------------------
# RUTAS DE DATOS
# ---------------------------------------------------------------------------
# Nota: Asegúrate de crear una carpeta llamada 'data/' en la raíz de tu proyecto
# y colocar allí los archivos CSV de la competición.
PATH_TRAIN_X = 'data/X_train.csv'
PATH_TRAIN_Y = 'data/y_train.csv'
PATH_TEST    = 'data/X_test.csv'

# Semilla y CV
RANDOM_STATE = 42
N_SPLITS     = 5

# Mejores hiperparámetros (obtenidos previamente con GridSearchCV / RandomizedSearchCV)
PARAMS_RF = {
    'n_estimators':      300,
    'max_depth':         30,
    'min_samples_split': 5,
    'min_samples_leaf':  2,
    'class_weight':      None,
}

PARAMS_LGBM = {
    'n_estimators':      800,
    'max_depth':         12,
    'learning_rate':     0.03,
    'num_leaves':        255,
    'subsample':         0.7,
    'colsample_bytree':  0.6,
    'reg_alpha':         0.1,
    'reg_lambda':        2.0,
    'min_child_samples': 30,
    'class_weight':      None,
}

PARAMS_XGB = {
    'n_estimators':      300,
    'max_depth':         10,
    'learning_rate':     0.1,
    'subsample':         0.8,
    'colsample_bytree':  0.7,
    'eval_metric':       'mlogloss',
}

ENSEMBLE_WEIGHTS = [3, 2, 2]   # RF, LGBM, XGB

# Mapeo de la variable objetivo
Y_MAP     = {'functional': 2, 'functional needs repair': 1, 'non functional': 0}
Y_MAP_INV = {2: 'functional', 1: 'functional needs repair', 0: 'non functional'}


## 3. MLflow — tracking de experimentos

In [ ]:
mlflow.set_experiment("Pump It Up - Íñigo Ochoa")
print("MLflow experiment configurado.")


## 4. Carga de datos

In [ ]:
data   = pd.read_csv(PATH_TRAIN_X, parse_dates=['date_recorded'])
target = pd.read_csv(PATH_TRAIN_Y)
test   = pd.read_csv(PATH_TEST,    parse_dates=['date_recorded'])

# Comprobaciones de integridad
assert data['id'].isin(target['id']).all(), "Hay ids de train sin etiqueta"
assert not data['id'].duplicated().any(),   "Hay ids duplicados en train"

# Unión X_train + y_train
df = pd.merge(data, target, on='id', how='left').rename(columns={'status_group': 'y'})

print(f"Train: {df.shape}  |  Test: {test.shape}")
df.head(3)


## 5. EDA — Análisis Exploratorio de Datos

El dataset cuenta con 40 variables iniciales (numéricas, categóricas y una fecha).  
Se detectan los siguientes patrones relevantes:

- **Missings implícitos:** `longitude`, `gps_height`, `population`, `construction_year` y `amount_tsh` tienen ceros que representan datos ausentes.
- **Alta cardinalidad:** `funder`, `installer`, `subvillage`, `ward`, `scheme_name`.
- **Desbalanceo de clases:** la clase *functional needs repair* (~7 %) es minoritaria.
- **Columnas redundantes:** `payment_type` ≡ `payment`; `quantity_group` ≡ `quantity`.


In [ ]:
df.info()


In [ ]:
df.describe().round(2).T


In [ ]:
df.describe(include='object').T


In [ ]:
print("Nulos por columna (solo las que tienen):")
print(df.isna().sum()[df.isna().sum() > 0].sort_values(ascending=False))


In [ ]:
# Distribución de la variable objetivo
fig, ax = plt.subplots(figsize=(6, 3))
df['y'].value_counts().plot(kind='bar', ax=ax, color=['#2ecc71', '#e67e22', '#e74c3c'])
ax.set_title('Distribución de la variable objetivo')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()
print(df['y'].value_counts(normalize=True).round(3))


In [ ]:
# Missings implícitos: longitude = 0 es imposible para Tanzania
sns.scatterplot(data=df, x='longitude', y='latitude', hue='y', alpha=0.4, s=5)
plt.title('longitude = 0 → missing implícito')
plt.show()


In [ ]:
# Variables duplicadas
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for col, ax in zip(['payment_type', 'payment'], axes):
    sns.histplot(data=df, x=col, hue='y', multiple='stack', ax=ax)
    ax.tick_params(axis='x', rotation=90)
fig.suptitle('payment_type y payment son equivalentes → se elimina payment_type')
plt.tight_layout()
plt.show()


In [ ]:
# Informe automático completo (tarda ~1 min)
# report = ProfileReport(df, title="Pump It Up EDA")
# report.to_notebook_iframe()


## 6. Limpieza de datos

| Decisión | Columna(s) | Motivo |
|---|---|---|
| Eliminar | `scheme_name` | >50% nulos + nombres propios |
| Eliminar | `num_private` | ~100% ceros, sin descripción |
| Eliminar | `recorded_by` | Un único valor → sin información |
| Eliminar | `payment_type` | Duplicado de `payment` |
| Eliminar | `quantity_group` | Duplicado de `quantity` |
| Imputar → 'Desconocido' | `scheme_management`, `installer` | La ausencia parece una categoría |
| Imputar → 0 | `permit` | Sin registro ≈ sin permiso |
| Mantener ceros | `longitude`, `gps_height`, `population` | El ensemble los maneja; imputar no mejoró el CV |


In [ ]:
def clean(df_: pd.DataFrame) -> pd.DataFrame:
    """Limpieza de datos. Aplicar igual a train y test."""
    df_ = df_.copy()

    # Eliminar columnas irrelevantes o redundantes
    drop_cols = ['scheme_name', 'num_private', 'recorded_by', 'payment_type', 'quantity_group']
    df_ = df_.drop(columns=[c for c in drop_cols if c in df_.columns])

    # Imputar nulos explícitos
    df_['scheme_management'] = df_['scheme_management'].fillna('Desconocido')
    df_['installer']         = df_['installer'].fillna('Desconocido')
    df_['permit']            = df_['permit'].fillna(0)

    return df_

df   = clean(df)
test = clean(test)
print(f"Train tras limpieza: {df.shape}  |  Test: {test.shape}")


## 7. Feature Engineering

### Variables activas

| Variable | Tipo | Descripción |
|---|---|---|
| `antiguedad` | Numérica | `date_recorded.year − construction_year` |
| `mes` | Numérica | Mes de la medición (estacionalidad) |
| `gps_height_600` | Binaria | 1 si altitud > 600 m |
| `buenas_reg` | Binaria | 1 si la región tiene < 33% de bombas no funcionales |
| `malos_funders` | Binaria | 1 si el financiador tiene ≥ 40% de bombas no funcionales |
| `lga_top_func` | Ordinal (0/1/2) | Clasificación del distrito por tasa funcional |

> **Nota sobre leakage:** `buenas_reg`, `malos_funders` y `lga_top_func` se calculan  
> con estadísticas de `y`. Para evitar leakage dentro del cross-validation, estas  
> transformaciones están encapsuladas en un `TransformerMixin` que se ajusta  
> **solo con los datos de cada fold de entrenamiento** y se aplica al fold de validación.


In [ ]:
def add_base_features(df_: pd.DataFrame) -> pd.DataFrame:
    """
    Features que NO dependen de y → se pueden calcular antes del CV
    sin riesgo de leakage.
    """"
    df_ = df_.copy()
    df_['antiguedad']    = df_['date_recorded'].dt.year - df_['construction_year']
    df_['mes']           = df_['date_recorded'].dt.month
    df_['gps_height_600'] = (df_['gps_height'] > 600).astype(int)
    df_ = df_.drop(columns='date_recorded')
    return df_

df   = add_base_features(df)
test = add_base_features(test)


In [ ]:
class TargetRatioFeatures(BaseEstimator, TransformerMixin):
    """
    Calcula features basadas en estadísticas de y (target-ratio encoding ligero).
    Al ser un TransformerMixin, fit() usa solo X_train del fold → sin leakage.

    Requiere que X tenga la columna 'y_num' durante el fit.
    En transform() crea las columnas derivadas usando únicamente los umbrales
    aprendidos en fit().
    """

    def __init__(self,
                 reg_threshold: float = 0.33,
                 funder_min_count: int = 200,
                 funder_nf_threshold: float = 0.40,
                 lga_top_threshold: float = 0.65,
                 lga_bot_threshold: float = 0.40):
        self.reg_threshold      = reg_threshold
        self.funder_min_count   = funder_min_count
        self.funder_nf_threshold = funder_nf_threshold
        self.lga_top_threshold  = lga_top_threshold
        self.lga_bot_threshold  = lga_bot_threshold

    def fit(self, X, y=None):
        # y debe estar en X como columna 'y_num' durante el fit
        Xy = X.copy()
        Xy['_y'] = y  # y numérico pasado explícitamente

        inv = {2: 'functional', 1: 'functional needs repair', 0: 'non functional'}
        Xy['_y_str'] = Xy['_y'].map(inv)

        # buenas_reg
        tabla_reg = (
            Xy.groupby('region')['_y_str']
            .value_counts(normalize=True)
            .unstack(fill_value=0)
        )
        self.buenas_reg_ = set(tabla_reg.index[tabla_reg.get('non functional', 0) < self.reg_threshold])

        # malos_funders
        counts = Xy['funder'].value_counts()
        top_funders = set(counts[counts >= self.funder_min_count].index)
        Xy['_funder_g'] = Xy['funder'].apply(lambda x: x if x in top_funders else 'Otros')
        tabla_funder = (
            Xy.groupby('_funder_g')['_y_str']
            .value_counts(normalize=True)
            .unstack(fill_value=0)
        )
        self.top_funders_    = top_funders
        self.malos_funders_  = set(tabla_funder.index[tabla_funder.get('non functional', 0) >= self.funder_nf_threshold])

        # lga_top_func
        tabla_lga = (
            Xy.groupby('lga')['_y_str']
            .value_counts(normalize=True)
            .unstack(fill_value=0)
        )
        self.top_lga_    = set(tabla_lga.index[tabla_lga.get('functional', 0) > self.lga_top_threshold])
        self.bottom_lga_ = set(tabla_lga.index[tabla_lga.get('functional', 0) < self.lga_bot_threshold])

        return self

    def transform(self, X):
        X_ = X.copy()

        X_['buenas_reg']   = X_['region'].apply(lambda x: 1 if x in self.buenas_reg_ else 0)

        X_['_funder_g']    = X_['funder'].apply(lambda x: x if x in self.top_funders_ else 'Otros')
        X_['malos_funders'] = X_['_funder_g'].apply(lambda x: 1 if x in self.malos_funders_ else 0)
        X_ = X_.drop(columns='_funder_g')

        X_['lga_top_func'] = X_['lga'].apply(
            lambda x: 2 if x in self.top_lga_ else (0 if x in self.bottom_lga_ else 1)
        )

        return X_


In [ ]:
# Visualizaciones que motivaron cada feature
# (usando datos de train completo — solo para exploración, no para selección de umbrales)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

sns.histplot(data=df.loc[df.construction_year != 0],
             x='antiguedad', hue='y', multiple='fill', ax=axes[0])
axes[0].set_title('Antigüedad vs estado')

sns.histplot(data=df, x='mes', hue='y', multiple='fill', ax=axes[1])
axes[1].set_title('Mes de registro vs estado')

sns.histplot(data=df, x='gps_height', hue='y', multiple='fill', ax=axes[2])
axes[2].axvline(600, color='red', linestyle='--', label='600m')
axes[2].set_title('Altitud vs estado')
axes[2].legend()

plt.tight_layout()
plt.show()


## 8. Codificación de variables categóricas

Se usa `LabelEncoder` para cada columna categórica.  
Las categorías desconocidas en test se reemplazan por `'Otros'` antes de transformar,  
evitando errores en producción.


In [ ]:
def encode_categoricals(df_train: pd.DataFrame,
                        df_test:  pd.DataFrame,
                        exclude:  list = None) -> tuple:
    """
    Ajusta LabelEncoders sobre df_train y los aplica a df_test.
    Categorías del test no vistas en train → 'Otros'.
    Devuelve (df_train_enc, df_test_enc, encoders_dict).
    """
    exclude = exclude or []
    col_cat = [
        c for c in df_train.select_dtypes(include='object').columns
        if c not in exclude
    ]

    encoders = {}
    df_tr = df_train.copy()
    df_te = df_test.copy()

    for col in col_cat:
        le = LabelEncoder()
        todas = list(df_tr[col].astype(str).unique()) + ['Otros']
        le.fit(todas)
        df_tr[col] = le.transform(df_tr[col].astype(str))

        known = set(le.classes_)
        df_te[col] = df_te[col].apply(lambda x: x if x in known else 'Otros')
        df_te[col] = le.transform(df_te[col].astype(str))

        encoders[col] = le

    return df_tr, df_te, encoders


df_enc, test_enc, encoders = encode_categoricals(df, test, exclude=['y'])
print("Codificación completada.")
print(f"Train: {df_enc.shape}  |  Test: {test_enc.shape}")


In [ ]:
# Variable objetivo numérica
df_enc['y_num'] = df_enc['y'].map(Y_MAP)

X    = df_enc.drop(columns=['id', 'y', 'y_num'])
y    = df_enc['y_num']
X_te = test_enc.drop(columns=['id'])

print(f"X: {X.shape}  |  y: {y.shape}  |  X_test: {X_te.shape}")
print(f"Distribución de y:\n{y.value_counts().sort_index()}")


## 9. Búsqueda de hiperparámetros

> **Nota:** esta sección puede tardar horas. Los mejores parámetros encontrados  
> ya están cargados en `PARAMS_RF`, `PARAMS_LGBM` y `PARAMS_XGB` en la sección de  
> configuración. Ejecutar solo si se quiere repetir la búsqueda.


In [ ]:
# ── Random Forest ─────────────────────────────────────────────────────────────
SEARCH = False   # ← cambiar a True para ejecutar

if SEARCH:
    grid_rf = {
        'n_estimators':      [300, 500],
        'max_depth':         [20, 30, None],
        'min_samples_split': [2, 5],
        'min_samples_leaf':  [1, 2],
        'class_weight':      [None, 'balanced'],
    }
    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    search_rf = GridSearchCV(
        RandomForestClassifier(random_state=RANDOM_STATE),
        param_grid=grid_rf, scoring='accuracy', cv=cv, verbose=2, n_jobs=-1
    )
    search_rf.fit(X, y)
    print(f"Mejores parámetros RF:  {search_rf.best_params_}")
    print(f"Mejor accuracy RF (CV): {search_rf.best_score_:.4f}")


In [ ]:
# ── LightGBM ──────────────────────────────────────────────────────────────────
if SEARCH:
    grid_lgbm = {
        'n_estimators':      [300, 500, 800, 1000],
        'max_depth':         [10, 12, 20, -1],
        'learning_rate':     [0.01, 0.03, 0.05, 0.1],
        'num_leaves':        [63, 127, 255],
        'subsample':         [0.7, 0.8, 0.9],
        'colsample_bytree':  [0.6, 0.7, 0.8],
        'reg_alpha':         [0, 0.1, 0.5],
        'reg_lambda':        [0.5, 1.0, 2.0],
        'min_child_samples': [10, 20, 30],
        'class_weight':      [None, 'balanced'],
    }
    search_lgbm = RandomizedSearchCV(
        LGBMClassifier(device='gpu', random_state=RANDOM_STATE, verbose=-1),
        param_distributions=grid_lgbm, n_iter=50,
        scoring='accuracy', cv=StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE),
        verbose=2, n_jobs=-1, random_state=RANDOM_STATE
    )
    search_lgbm.fit(X, y)
    print(f"Mejores parámetros LGBM:  {search_lgbm.best_params_}")
    print(f"Mejor accuracy LGBM (CV): {search_lgbm.best_score_:.4f}")


In [ ]:
# ── XGBoost ───────────────────────────────────────────────────────────────────
if SEARCH:
    grid_xgb = {
        'n_estimators':     [300, 500, 1000],
        'max_depth':        [6, 8, 10],
        'learning_rate':    [0.01, 0.05, 0.1],
        'subsample':        [0.7, 0.8, 1.0],
        'colsample_bytree': [0.7, 0.8, 1.0],
    }
    search_xgb = GridSearchCV(
        XGBClassifier(device='cuda', eval_metric='mlogloss', random_state=RANDOM_STATE),
        param_grid=grid_xgb, scoring='accuracy',
        cv=StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE),
        verbose=2, n_jobs=-1
    )
    search_xgb.fit(X, y)
    print(f"Mejores parámetros XGB:  {search_xgb.best_params_}")
    print(f"Mejor accuracy XGB (CV): {search_xgb.best_score_:.4f}")


## 10. Modelo final — Ensemble de Soft Voting

Combinamos tres modelos mediante `VotingClassifier` con `voting='soft'`:

- **Random Forest** (peso 3): mejor rendimiento individual en CV  
- **LightGBM** (peso 2): rápido, maneja bien variables categóricas y missings  
- **XGBoost** (peso 2): complementa a LGBM con diferente sesgo de inducción  

Los pesos se ajustaron empíricamente comparando runs en MLflow.


In [ ]:
rf_model   = RandomForestClassifier(**PARAMS_RF,   random_state=RANDOM_STATE, n_jobs=-1)
lgbm_model = LGBMClassifier(**PARAMS_LGBM, device='cpu', random_state=RANDOM_STATE,
                             n_jobs=-1, verbose=-1)
xgb_model  = XGBClassifier(**PARAMS_XGB, device='cpu',
                            random_state=RANDOM_STATE, n_jobs=-1)

ensemble = VotingClassifier(
    estimators=[
        ('random_forest', rf_model),
        ('lightgbm',      lgbm_model),
        ('xgboost',       xgb_model),
    ],
    voting='soft',
    weights=ENSEMBLE_WEIGHTS,
    n_jobs=-1
)


## 11. Validación cruzada — sin leakage

`TargetRatioFeatures` se ajusta dentro de cada fold con los datos de entrenamiento  
de ese fold únicamente, y se aplica al fold de validación. Esto garantiza que las  
estadísticas de `buenas_reg`, `malos_funders` y `lga_top_func` no contaminen la  
evaluación.


In [ ]:
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

fold_scores = []
all_preds   = np.zeros(len(y), dtype=int)

trf = TargetRatioFeatures()

for fold, (train_idx, val_idx) in enumerate(cv.split(X, y), 1):
    X_tr_fold, X_val_fold = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
    y_tr_fold, y_val_fold = y.iloc[train_idx],        y.iloc[val_idx]

    # Fit de features con target → solo sobre train del fold
    trf.fit(X_tr_fold, y_tr_fold)
    X_tr_fold  = trf.transform(X_tr_fold)
    X_val_fold = trf.transform(X_val_fold)

    ensemble.fit(X_tr_fold, y_tr_fold)
    preds = ensemble.predict(X_val_fold)
    all_preds[val_idx] = preds

    score = accuracy_score(y_val_fold, preds)
    fold_scores.append(score)
    print(f"  Fold {fold}: {score:.4f}")

mean_cv = np.mean(fold_scores)
std_cv  = np.std(fold_scores)
print(f"\nCV Accuracy:  {mean_cv:.4f} ± {std_cv:.4f}")


In [ ]:
# Accuracy por clase
cm = confusion_matrix(y, all_preds)
acc_por_clase = cm.diagonal() / cm.sum(axis=1)
labels = ['non functional (0)', 'functional needs repair (1)', 'functional (2)']
print("Accuracy por clase:")
for label, acc in zip(labels, acc_por_clase):
    print(f"  {label}: {acc:.4f}")

print()
print(classification_report(y, all_preds, target_names=['non functional', 'functional needs repair', 'functional']))


In [ ]:
# Matriz de confusión visual
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['non func.', 'needs repair', 'functional'],
            yticklabels=['non func.', 'needs repair', 'functional'], ax=ax)
ax.set_xlabel('Predicho')
ax.set_ylabel('Real')
ax.set_title('Matriz de confusión — CV (5 folds)')
plt.tight_layout()
plt.show()


## 12. Registro del experimento en MLflow

In [ ]:
with mlflow.start_run(run_name="Ensemble RF+LGBM+XGB (3-2-2) — leakage-free"):
    mlflow.log_params({
        **{f'rf_{k}':   v for k, v in PARAMS_RF.items()},
        **{f'lgbm_{k}': v for k, v in PARAMS_LGBM.items()},
        **{f'xgb_{k}':  v for k, v in PARAMS_XGB.items()},
        'ensemble_weights': str(ENSEMBLE_WEIGHTS),
        'voting':    'soft',
        'n_splits':  N_SPLITS,
        'fe_leakage_free': True,
    })
    mlflow.log_metric('cv_accuracy', mean_cv)
    mlflow.log_metric('cv_std',      std_cv)

    # Entrenamos sobre todos los datos para el modelo final
    trf_final = TargetRatioFeatures()
    trf_final.fit(X, y)
    X_full = trf_final.transform(X)
    ensemble.fit(X_full, y)

    mlflow.sklearn.log_model(ensemble, 'modelo_ensemble')
    print(f"Run registrado. CV accuracy: {mean_cv:.4f} ± {std_cv:.4f}")


In [ ]:
# Comparativa de runs
runs = mlflow.search_runs(experiment_names=["Pump It Up - Íñigo Ochoa"])
runs[['tags.mlflow.runName', 'metrics.cv_accuracy', 'metrics.cv_std']].sort_values(
    'metrics.cv_accuracy', ascending=False
)


## 13. Generación del archivo de submission

In [ ]:
# Aplicar el mismo FE al test (usando trf_final ajustado con todos los datos de train)
X_te_fe = trf_final.transform(X_te)

y_test_pred      = ensemble.predict(X_te_fe)
y_test_pred_text = pd.Series(y_test_pred).map(Y_MAP_INV)

submission = pd.DataFrame({
    'id':           test_enc['id'],
    'status_group': y_test_pred_text
})

print("Distribución de predicciones:")
print(submission['status_group'].value_counts())
print()
print(submission.head())

submission.to_csv('submission_pump_it_up.csv', index=False)
print("\n✓ Archivo guardado: submission_pump_it_up.csv")


## 14. Resultados finales

| Métrica | Valor |
|---|---|
| **Accuracy en leaderboard** | **0.8186** |
| **Posición** | **2624 / 19 775** (~top 13%) |
| CV Accuracy (5 folds, leakage-free) | ver celda de validación |

### Accuracy por clase (CV)
La clase *functional needs repair* es la más difícil de predecir,  
lo que es esperable dado su bajo volumen (~7% del dataset).

### Decisiones clave
- **No imputar missings implícitos** (longitude=0, gps_height≤0): el ensemble  
  los gestiona internamente mejor que cualquier imputación probada.
- **class_weight=None** en RF y LGBM: a pesar del desbalanceo,  
  usar pesos balanceados redujo el accuracy en CV.
- **Peso mayor a RF (3 vs 2)**: RF obtuvo mejor accuracy individual  
  que LGBM y XGBoost por separado en este dataset.
